# 🏠 Real Estate Hedonic Pricing — End-to-End Pipeline

> 한국 아파트 실거래 데이터 → 공간 헤도닉 모형 추정 → 공간자기상관 보정 → 결과 해석

**Notebook 구성**:
1. 환경 설정 및 데이터 수집 (MOLIT API)
2. 지오코딩 (Kakao Local API)
3. 전처리 및 파생변수
4. 공간가중행렬 생성
5. 4개 모형 추정 (OLS → SLM → SEM → SDM)
6. 진단 및 모형 선택
7. 시각화 및 해석

> ⚠️ 본 노트북은 **스켈레톤** 상태입니다. 각 셀의 `NotImplementedError`는 구현 가이드로 이해하세요.

## 1. 환경 설정

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('../.env')

# 모듈 로드
from hedonic.molit_api import MolitClient
from hedonic.geocoding import KakaoGeocoder
from hedonic.preprocessing import full_pipeline
from hedonic.weights import build_weights, summarize_weights
from hedonic.models import SpatialHedonicModel, compare_models
from hedonic.diagnostics import morans_i, local_moran, lm_diagnostics
from hedonic.visualization import plot_price_choropleth, plot_lisa_cluster

## 2. MOLIT 실거래가 수집

**Target**: 강남구(11680), 2026-01 ~ 2026-03

In [ ]:
client = MolitClient()
raw_df = client.fetch_multi_period(
    region_codes=['11680'],
    year_months=['202601', '202602', '202603'],
)
print(f'Raw transactions: {len(raw_df):,}건')
raw_df.head()

## 3. 지오코딩

Kakao Local API로 주소 → 위경도 변환. 캐시가 걸려 있어 동일 주소는 1회만 호출됩니다.

In [ ]:
geo = KakaoGeocoder()
geocoded = geo.geocode_dataframe(raw_df, address_col='full_address')
print(f'지오코딩 성공: {geocoded.dropna(subset=["longitude"]).shape[0]:,}건')
print(f'실패: {len(geo.failed_addresses):,}건')

## 4. 전처리

- 해제거래 제거
- IQR 기반 이상치 제거
- 파생변수 (㎡당 가격, 건물 연령, 층 범주, 로그변환)
- 공간조인 (행정동 경계)

In [ ]:
dong_boundary = gpd.read_file('../data/raw/seoul_dong.geojson')
ads = full_pipeline(geocoded, dong_boundary=dong_boundary)
print(f'최종 분석 데이터셋: {len(ads):,}건 × {ads.shape[1]}개 변수')
ads[['deal_amount', 'area', 'price_per_sqm', 'building_age', 'log_price']].describe()

## 5. 공간가중행렬

Point 데이터이므로 KNN (k=8)을 기본으로 사용.

In [ ]:
w_knn = build_weights(ads, method='knn', k=8)
print(summarize_weights(w_knn))
# {'n': ..., 'mean_neighbors': 8.0, 'pct_islands': 0.0, 'sparsity': ...}

## 6. OLS 베이스라인 + 공간자기상관 진단

In [ ]:
X_vars = ['log_area', 'building_age', 'floor']
ols_model = SpatialHedonicModel(ads, y='log_price', X=X_vars, method='OLS')
ols_res = ols_model.fit()
print(ols_res.summary())

# OLS 잔차의 Moran's I 검정
mi = morans_i(ols_res.residuals, w_knn)
print(f"Moran's I = {mi.statistic:.3f} (p={mi.p_value:.4f})")
# → p < 0.05면 공간자기상관 있음 → SEM/SLM 필요

## 7. LM 진단으로 모형 선택

**Anselin & Florax (1995) 규칙**:
- 둘 다 유의하지 않음 → OLS
- 하나만 유의 → 해당 모형
- 둘 다 유의 → Robust LM 비교

In [ ]:
lm = lm_diagnostics(ols_res, w_knn)
print(f'LM-Lag   = {lm.lm_lag:.2f} (p={lm.lm_lag_p:.4f})')
print(f'LM-Error = {lm.lm_error:.2f} (p={lm.lm_error_p:.4f})')
print(f'추천 모형: {lm.recommend()}')

## 8. 4개 모형 추정 및 비교

In [ ]:
slm_res = SpatialHedonicModel(ads, 'log_price', X_vars, w_knn, 'SLM').fit()
sem_res = SpatialHedonicModel(ads, 'log_price', X_vars, w_knn, 'SEM').fit()
sdm_res = SpatialHedonicModel(ads, 'log_price', X_vars, w_knn, 'SDM').fit()

comparison = compare_models([ols_res, slm_res, sem_res, sdm_res])
comparison

## 9. LISA 핫스팟·콜드스팟 식별

In [ ]:
lisa_df = local_moran(ads['price_per_sqm'].values, w_knn)

fig, ax = plt.subplots(figsize=(12, 10))
plot_lisa_cluster(ads, lisa_df, ax=ax)
ax.set_title('㎡당 가격 LISA 클러스터 (강남구 2026 Q1)', fontsize=14)
plt.savefig('../results/figures/lisa_cluster.png', dpi=150, bbox_inches='tight')

## 10. 결과 해석

**Implicit Price 추출 예시** (SEM 기준):

- `log_area` 계수 β = 0.85 → 면적 1% ↑ = 가격 0.85% ↑ (규모의 경제)
- `building_age` 계수 β = -0.012 → 1년 노후화 = 가격 1.2% ↓
- `floor` 계수 β = 0.008 → 1층 상승 = 가격 0.8% ↑
- `λ` (공간 error) ≈ 0.65 → 관측되지 않은 공간 요인이 강하게 존재

**정책적 시사점**:
- 공간자기상관 보정 전후 면적 계수가 유의하게 달라진다면,
  OLS 기반 감정평가는 체계적 편향을 가짐 → SEM 표준 채택 필요.